## Тетрадка

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # читает .env из текущей папки

True

In [10]:
SYSTEM = """Ты — senior data engineer, ментор для middle-специалиста.
Правила:
1. Отвечай кратко: не более 5 предложений без воды.
2. Сначала суть, потом нюанс — в таком порядке.
3. Если в вопросе есть неточность — сначала поправь её, потом отвечай.
4. Без эмодзи и маркетингового тона.
"""

In [7]:
from openai import OpenAI


In [11]:

client = OpenAI(
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    max_retries=2,          # встроенный retry — помнишь 0.4.6
)
MODEL = os.environ["LLM_MODEL"]

QUESTION = "Расскажи про оконные функции pyspark"

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": QUESTION},
    ],
    temperature=0,
    max_tokens=1500,
)

print("=== ОТВЕТ ===")
print(resp.choices[0].message.content)
print("\n=== МЕТА ===")
print("finish_reason:", resp.choices[0].finish_reason)
print("модель ответила:", resp.model)
print("usage:", resp.usage)

=== ОТВЕТ ===
Оконные функции в PySpark задаются через `Window.partitionBy(...).orderBy(...)` и применяются к колонке через `F.row_number().over(w)`, `F.sum("x").over(w)`, `F.lag("x", 1).over(w)` и т.п.: окно не сворачивает строки, как `groupBy`, а добавляет вычисленное значение к каждой строке. Для накопительных агрегатов нужен явный фрейм (`rowsBetween(Window.unboundedPreceding, Window.currentRow)`), иначе при наличии `orderBy` по умолчанию берётся расширяющееся окно, а без него — всё партишн целиком. Нюанс производительности: `partitionBy` вызывает shuffle, а `Window` без `partitionBy` собирает все данные в одну партицию, поэтому по возможности сортируйте и партиционируйте по одним и тем же ключам и не тащите в окно лишние колонки.

=== МЕТА ===
finish_reason: stop
модель ответила: deepseek/deepseek-v4.1-flash-1017
usage: CompletionUsage(completion_tokens=362, prompt_tokens=133, total_tokens=495, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_t